### Практика: большие языковые модели и их возможности

<!-- ![img](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fbucketeer-e05bbc84-baa3-437e-9518-adb32be77984.s3.amazonaws.com%2Fpublic%2Fimages%2F4470ce74-e595-4750-92a5-5f21f040df6d_577x432.jpeg) -->
![img](https://i.imgur.com/QGYa2J8.jpeg)

В этом ноутбуке ты поэкспериментируешь с некоторыми из самых больших языковых моделей, доступных в интернете.

_Основано на работах Tim Dettmers, Ruslan Svirschevsky, Artem Chumachenko, Younes Belkada, Felix Marty, Yulian Gilyazev, Gosha Zolotov, Andrey Ishutin, Elena Volf, Artemiy Vishnyakov, Svetlana Shirokovskih, Maria Ivanova._

> Неофициальный перевод [оригинала](./homework_ICL.ipynb). Код сохранён.

### Часть 1. Промпт-инжиниринг (всего 4 балла)

Твоя задача — заставить **базовую** языковую модель решать несколько задач с помощью промптов.

Базовая модель умеет **продолжать текст**, но её не обучали следовать инструкциям пользователя. Направлять её поведение нужно продуманными промптами и примерами few-shot.

**Какую модель использовать?** Запусти базовую модель локально, например [`Qwen/Qwen3-4B-Base`](https://huggingface.co/Qwen/Qwen3-4B-Base).

**Что сдать:** для каждой из четырёх задач покажи свои **промпты**, **код запуска**, **выход модели** и короткое **описание решения**.

**Как ведёт себя базовая модель:** она продолжает то, что ты написал. Начни диалог — она продолжит диалог; приведи несколько готовых примеров — она подражает их формату.

Загрузи модель в двух ячейках ниже и используй предоставленную функцию `generate()`.

**Повторять и исправлять можно.** Базовые модели чувствительны к формулировкам. Если ответ не подходит, измени промпт или добавь few-shot-пример. Если для одной задачи требуется слишком много попыток, скорее всего, промпт стоит пересмотреть.

In [ ]:
%pip install -q "transformers>=5.0" "accelerate>=1.0"
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm, trange

device = 'cuda'
major, _ = torch.cuda.get_device_capability()
dtype = torch.bfloat16 if major >= 8 else torch.float16   # T4 (compute 7.5) has no native bf16 -> use fp16
print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(), "| dtype:", dtype)

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device).eval()


def generate(prompt, max_new_tokens=64, **kwargs):
    """Continue `prompt` with the base model; return only the newly generated text.
    Pass do_sample=True (+ temperature/top_p/top_k) for creative output, or nothing for greedy."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, **kwargs)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


# быстрая проверка: базовая модель просто продолжает текст
print(generate("Once upon a time, deep in the forest,", max_new_tokens=40,
               do_sample=True, temperature=0.8, top_p=0.9))

**Задание 1 (1 балл):** организуй беседу между любыми двумя участниками из следующих категорий:

- Известный человек или политик по твоему выбору.
- Любой вымышленный персонаж (кроме Дарта Вейдера).
- Ты сам.

Сравни два подхода: **а)** указать только имена персонажей; **б)** предоставить дополнительную информацию (как в примере).

**Пример:** Тони разговаривает с Дартом Вейдером. Чёрный текст написан вручную, синий сгенерирован.

![img](https://i.imgur.com/a1QhKF7.png)

In [ ]:
# <your code here>

**Выбери задание 2a или 2b (1 балл)** в зависимости от своей модели. Можно выполнить оба, но балл начисляется только за одно.

**Задание 2a: zero-shot перевод.** Возьми первую строфу стихотворения Эдгара Аллана По [«Ворон»](https://www.poetryfoundation.org/poems/48860/the-raven) и **переведи её на французский**. Разрешено взять любой другой текст не меньшего объёма.

Исходный текст (оставлен на английском как вход для упражнения):

```
Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
“’Tis some visitor,” I muttered, “tapping at my chamber door—
            Only this and nothing more.”
```

Проверь качество, переведя полученный французский текст **обратно на английский** через публичный сервис машинного перевода.

**Задание 2b: классификация токсичности** по датасету [SetFit/toxic_conversations](https://huggingface.co/datasets/SetFit/toxic_conversations).

Настрой бинарную классификацию (токсичный/нетоксичный) в режиме few-shot. В примерах используй 2–3 токсичных и 2–3 нетоксичных текста. Измерь accuracy на **не менее 25 примерах**. Возможно, придётся перебрать несколько промптов, прежде чем найдётся работающий.

In [ ]:
# <your code here>

**Задание 3 (1 балл):** составь промпт и few-shot-примеры, с помощью которых модель будет **менять гендерные местоимения** главного действующего лица предложения в выбранном направлении. Например, `the doctor took off his mask` ↔ `the doctor took off her mask`.

In [ ]:
# <your code here>

**Задание 4 (1 балл):** напиши промпт и примеры, чтобы модель **переводила имперские единицы в метрические** (мили → километры, мили/час → км/ч). Модель должна переписывать заданное предложение, заменяя все имперские единицы их метрическими эквивалентами.

Когда заработает на простых расстояниях и скоростях, придумай сложные случаи, где модель **ошибается**.

Обрати внимание: **1 миля не равна 1 километру**.

In [ ]:
# <your code here>

### Часть 2. Локальный инференс и внутреннее устройство генерации текста

В первой части мы использовали `model.generate` как чёрный ящик. Теперь заглянем **под капот**: как языковая модель превращает логиты в текст и как самостоятельно реализовать декодирование.

Выше уже загружена [Qwen/Qwen3-4B-Base](https://huggingface.co/Qwen/Qwen3-4B-Base) — плотная 4B-модель, помещающаяся на бесплатную Colab T4 в fp16 (около 8 ГБ).

## Генерация текста

**Сравнение стратегий генерации языковой модели:**

| Стратегия | Описание | Плюсы и минусы |
| --- | --- | --- |
| Greedy Search | Всегда выбирает самый вероятный следующий токен. | **Плюсы:** просто и быстро. **Минусы:** возможны повторы и несвязный текст. |
| Sampling with Temperature | Вносит случайность в выбор токена; высокая температура усиливает её. | **Плюсы:** разнообразие. **Минусы:** при высокой температуре возможен бессмысленный результат. |
| Nucleus Sampling (Top-p) | Выбирает токен из минимального набора кандидатов с суммарной вероятностью выше заданного порога p. | **Плюсы:** баланс качества и разнообразия. **Минусы:** подбор p неочевиден. |
| Beam Search | На каждом шаге поддерживает k самых перспективных гипотез. | **Плюсы:** часто надёжнее жадного поиска. **Минусы:** разнообразие ограничено, ответы могут быть шаблонными. |
| Top-k Sampling | Случайно выбирает из k самых вероятных токенов. | **Плюсы:** разнообразнее. **Минусы:** результат иногда менее связен. |
| Length Normalization | Делит логарифмы вероятностей на длину последовательности в заданной степени, ослабляя предпочтение коротким гипотезам. | **Плюсы:** можно генерировать более информативные ответы. **Минусы:** нужно настраивать коэффициент. |
| Stochastic Beam Search | Добавляет случайность при отборе k гипотез. | **Плюсы:** повышает разнообразие. **Минусы:** непросто выбрать баланс с качеством. |
| Minimum Bayes Risk (MBR) | Выбирает гипотезу с наименьшей ожидаемой потерей по выбранной функции качества. | **Плюсы:** оптимизирует нужную метрику. **Минусы:** сложные вычисления и требования к функции потерь. |

Документация:
- [`AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate).
- [`AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode).
- [Стратегии генерации в Hugging Face](https://huggingface.co/docs/transformers/generation_strategies).

### Генерация с Hugging Face

In [ ]:
prompt = 'The first discovered martian lifeform looks like'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
print("Input batch (encoded):", batch)

output_tokens = model.generate(**batch, max_new_tokens=64, do_sample=True, temperature=0.8)
# жадное декодирование:                                     do_sample=False)
# лучевой поиск наиболее вероятной последовательности:       num_beams=4)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))

#### Низкоуровневая реализация генерации текста

In [ ]:
prompt = "Moscow is the capital of"
# prompt = "Skippy, a young android, likes to dream about electric"

print(prompt, '\n')

voc = tokenizer.get_vocab()
voc_rev = {v:k for k, v in voc.items()}  # reverse vocab for decode

for i in range(10):
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    next_token_id = torch.multinomial(probs.flatten(), num_samples=1)

    next_token = tokenizer.decode(next_token_id)
    prompt += next_token

    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    top_tokens = sorted_indices[:5]
    print(f"Step #{i} candidates:")
    for t, p in zip (top_tokens, sorted_probs):
        t = voc_rev[t.item()]
        print(f"{t:<10}: {p:.4f} ")

    print(f'\nChosen token: {next_token}', end='\n\n', flush=True)

**Задание 5: реализуй nucleus sampling (2 балла).**

Используй шаблон `nucleus_sampling()` ниже. В качестве ориентира изучи низкоуровневый код генерации выше. **Не используй `model.generate`.**

**Бонус: реализуй beam search (3 дополнительных балла).**

In [ ]:
from typing import Tuple, List
def nucleus_sampling(model, tokenizer, prompt: str, prob: float = 0.5) -> Tuple[str, List[str]]:
    """generates the next token from the nucleus of tokens with cumulative probability up to param:prob"""

    <YOUR CODE HERE>

    # sampled_token — строка с сгенерированным токеном
    # possible_tokens — список всех токенов с ненулевой вероятностью
    return sampled_token, possible_tokens

In [ ]:
# Проверки nucleus sampling.
prompt = "Elbrus is the highest"

sampled, nucleus = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
print("sampled:", repr(sampled), "| nucleus size:", len(nucleus))

# 1) the sampled token must come from inside the nucleus
assert sampled in nucleus
assert len(nucleus) >= 1

# 2) the nucleus depends only on `prob`, not on the random draw -> it is deterministic across calls
_, nucleus_again = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
assert sorted(nucleus) == sorted(nucleus_again), "the nucleus itself should not be random"

# 3) a larger probability threshold never yields a smaller nucleus
_, small = nucleus_sampling(model, tokenizer, prompt, prob=0.1)
_, large = nucleus_sampling(model, tokenizer, prompt, prob=0.95)
assert len(large) >= len(small), "higher p should include at least as many tokens"

# 4) a tiny threshold collapses to the single most likely token
_, top1 = nucleus_sampling(model, tokenizer, prompt, prob=1e-6)
assert len(top1) == 1, "with p -> 0 the nucleus is just the top-1 token"

print("All invariant checks passed. (Also eyeball that the nucleus tokens are sensible continuations.)")

### Часть 3. Промптинг с цепочкой рассуждений (Chain-of-Thought, всего 4 балла)

![img](https://github.com/kojima-takeshi188/zero_shot_cot/raw/main/img/image_stepbystep.png)

---

In [ ]:
import json
import random
import locale; locale.getpreferredencoding = lambda: "UTF-8"
!wget https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json -O aqua.json
data = list(map(json.loads, open("aqua.json")))

In [ ]:
print("Example:")
data[150]

### Наивное решение

Предложим модели выбрать ответ для примера выше (`data[150]`) из указанных вариантов. Используем формат, напоминающий решения из школьного задачника.

Обрати внимание: варианты ответа немного изменены по форматированию — добавлены пробел и открывающая скобка. Неизвестно, насколько это важно.

In [ ]:
EXAMPLE_0SHOT = """
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
""".strip()

In [ ]:
# непосредственное решение уравнения
batch = tokenizer(EXAMPLE_0SHOT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_0SHOT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))

Вот как решать аналогичную задачу с помощью **few-shot Chain-of-Thought**.

Нужно изменить три вещи:
- Добавить поле **Rationale**, содержащее пошаговое решение.
- Добавить несколько few-shot-примеров ранее решённых задач **с объяснениями (rationales)**.
- Изменить итоговый промпт, чтобы модель сначала выдавала рассуждение, а затем ответ.

In [ ]:
EXAMPLE_3SHOT_CHAIN_OF_THOUGHT = """
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;\noriginal price = 100*1.6 = 160;\nactual price = 160*0.8 = 128.\nAnswer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125\nNumber of bags sold = 3000/125 = 24\nAnswer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If the percentage of black marbles pulled out the second time represents their percentage in the bag, how many marbles in total Q does the bag currently hold?
Answer Choices: (A) 40 (B) 200 (C) 380 (D) 400 (E) 3200
Rationale: We know that there are 20 black marbles in the bag and this number represent 1/20 th of the number of all marbles in the bag, thus there are total Q of 20*20=400 marbles.\nAnswer: D.
Correct Answer: D


Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Rationale:
""".strip()

In [ ]:
batch = tokenizer(EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_3SHOT_CHAIN_OF_THOUGHT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))
#### NOTE: scroll down for the final answer (below the ======= line)

**Задание 6 (1 балл):** напиши функцию, которая автоматически формирует Chain-of-Thought-промпты. Следуй требованиям в docstring функции.

In [ ]:
QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

def make_prompt(*, main_question, fewshot_examples):
    """
    Your goal is to produce the same prompt as the EXAMPLE_3SHOT_CHAIN_OF_THOUGHT automatically

    For each few-shot question, make sure to follow the following rules:
    1. Each question begins with QUESTION_PREFIX, after which you should print the question without leading/traiiling spaces (if any)
    2. After the question, provide space-separated options. Each option should be put in double brackets, followed by option text, e.g. "(A) 146%"
    3. Then, provide the answer as a single letter (A-E)
    4. Finally, add trailing newlines from FEWSHOT_SEPARATOR

    Your final prompt should contain all fewshot_examples (in order), separated with FEWSHOT_SEPARATOR, then follow with main_question.
    The main_question should contain the question and options formatted the same way as in FEWSHOT_EXAMPLES.
    After that, you should prompt the model to produce an explanation (rationale) for the answer.

    Please make sure your prompt contains no leading/trailing newlines or spaces, same as in EXAMPLE_3SHOT_CHAIN_OF_THOUGHT
    """

    <YOUR CODE HERE>

    return <a string that contains the prompt formatted as per instructions above>



generated_fewshot_prompt = make_prompt(main_question=data[150], fewshot_examples=(data[30], data[20], data[5]))
assert generated_fewshot_prompt == EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, "prompts don't match"
assert generated_fewshot_prompt != make_prompt(main_question=data[150], fewshot_examples=())
assert generated_fewshot_prompt.endswith(make_prompt(main_question=data[150], fewshot_examples=()))

print("Well done!")

# подсказка: сравни два промпта через https://www.diffchecker.com или аналогичный инструмент

**Задание 7 (1 балл):** оцени качество промпта.

Запусти модель **на всём датасете** и измерь accuracy. Для каждого вопроса случайно выбирай **n=5 других вопросов** как few-shot-примеры. Убедись, что сам `main_question` не попал в примеры.

Для строго научной оценки данные принято разделять на набор для evaluation и отдельный набор few-shot-примеров. Здесь это желательно, но необязательно.

Сложная часть — правильно остановить генерацию. Без ограничения модель может сгенерировать новый вопрос и сама же на него ответить. Поэтому останавливай генерацию **сразу после появления `Final Answer: [A-E]`**.

Реализовать это можно вручную (по низкоуровневому примеру выше) либо через [stopping criteria в Transformers](https://discuss.huggingface.co/t/implimentation-of-stopping-criteria-list/20040/2).

Если всё сделано правильно, результат должен быть заметно лучше случайного. Однако не ожидай чудес: это далеко не лучшая модель, и она существенно уступит среднему человеку.

In [ ]:
NUM_SAMPLES = 0    # use this to count how many samples you evaluated
NUM_RESPONDED = 0  # how many times did the model produce Correct Answer: (letter) in it's response. use as a sanity check.
NUM_CORRECT = 0    # how many times did the model's chosen answer (letter) match the correct answer

In [ ]:
< A whole lot of your code here >

# необязательно: для ускорения обрабатывай несколько предложений одним батчем;
# при батчевой обработке убедись, что результаты совпадают с batch=1 при жадном декодировании

In [ ]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

if NUM_RESPONDED / NUM_SAMPLES < 0.9:
  print("Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.")
  print("Make sure you generate enough tokens that the model can produce a correct answer.")
  print("When in doubt, take a look at the full model output. You can often spot errors there.")

**Задание 8 (2 балла): эксперимент.**

Используй только что созданный тестовый стенд, чтобы ответить **на один** из следующих исследовательских вопросов.

### Вариант 1. Сколько few-shot-примеров необходимо?

Как accuracy зависит от числа примеров в промпте?

a. Измерь изменение accuracy при увеличении/уменьшении числа shots.

b. Попробуй добиться качественного **пошагового объяснения без примеров** (zero-shot).

В zero-shot разрешено экспериментировать с промптами и менять процедуру инференса.

### Вариант 2. Насколько надёжен такой способ промптинга?

_По мотивам исследования Антона Воронова, Лены Вольф и Макса Рябинина._

Проверь устойчивость поведения модели и accuracy к небольшим изменениям промпта.

a. Упадёт ли точность, если few-shot-примеры будут содержать **неправильные ответы**? Если объяснение содержит правильный ответ в конце, тоже исправь его.

b. Что изменится, если заменить названия вопроса/ответа на «Q» и «A», разместить их в одной строке или поменять разделители few-shot-примеров?

### Вариант 3. Важность метода инференса

Разные способы декодирования дают разные результаты.

a. Сравни greedy inference и beam search по качеству генерации.

b. Реализуй и оцени **выбор ответа голосованием (sampling with voting)**.

Алгоритм голосования: сначала сгенерируй k вариантов ответа (например, 50) через nucleus sampling или похожий метод. Затем посчитай, сколько вариантов выбрало каждую итоговую букву (A, B и т. д.). Побеждает вариант с наибольшим количеством голосов.

Для ускорения можно генерировать попытки параллельно батчем — передай `model.generate` список, содержащий много копий одного промпта.

---

**Общие правила:** для выбранного варианта нужно проверить **обе гипотезы (a и b)**. Можно заменить одну собственной идеей, но для получения всех баллов заранее согласуй это с преподавателями в Telegram.

Организуй код и отчёт по своему усмотрению, но он должен быть читаемым и исполняться **сверху вниз**.

Напиши короткий неформальный отчёт об экспериментах и результатах — **минимум два абзаца**, можно больше. Визуализации приветствуются.

ИИ разрешено использовать для помощи в составлении отчёта, но итоговый текст должен оставаться понятным человеку.

In [ ]:
# организуй своё решение удобным способом